In [1]:
# Genre classification using RF

In [2]:
import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score
from sklearn.inspection import permutation_importance
from sklearn.model_selection import RepeatedStratifiedKFold

In [3]:
# load genre labels 

df_verse_labels = pd.read_csv('TAPA Paper_ Genre Labels - Verse.csv')
df_prose_labels = pd.read_csv('TAPA Paper_ Genre Labels - Prose.csv')
df_verse_labels.head()

,Author,Work,Genre,File Name
0,Plautus,Amphitruo,drama,plautus.amphitruo.txt
1,Plautus,Asinaria,drama,plautus.asinaria.txt
2,Plautus,Aulularia,drama,plautus.aulularia.txt
3,Plautus,Bacchides,drama,plautus.bacchides.txt
4,Plautus,Captivi,drama,plautus.captivi.txt


In [4]:
df_prose_labels.head()

,Author,Work,Genre,File Name
0,Augustine,Epistulae (1-10),epistolography,augustine.epistulae_1_10.part.1.txt
1,Augustine,Epistulae (11-20),epistolography,augustine.epistulae_11_20.part.2.txt
2,Augustine,Epistulae (21-30),epistolography,augustine.epistulae_21_30.part.3.txt
3,Augustine,Epistulae (31-40),epistolography,augustine.epistulae_31_40.part.4.txt
4,Augustine,Epistulae (41-50),epistolography,augustine.epistulae_41_50.part.5.txt


In [5]:
# load stylometric data 

df_verse_data = pd.read_csv('20250922_172341-raw-verse.csv')
df_verse_data = df_verse_data.rename(columns={'Unnamed: 0': 'File Name'}) 
df_prose_data = pd.read_csv('20250922_174146-raw-prose.csv')
df_prose_data = df_prose_data.rename(columns={'Unnamed: 0': 'File Name'}) 

df_verse_data.head()

,File Name,word_count,sentence_count,sentence_length,fraction_sentence_relative,relative_clause_length,alius,antequam,atque_consonant,conjunction,...,personal,preposition,priusquam,quidam,quin,quominus,reflexive,si,superlative,ut
0,anonymous.laudes_domini.txt,913,42,21.738095,0.357143,8.772727,0,0,1,86,...,29,19,0,0,0,0,4,3,0,7
1,catullus.carmina.64.txt,2445,109,22.431193,0.357798,8.140351,3,0,0,130,...,58,78,0,0,0,0,12,2,1,16
2,catullus.carmina.65_116.txt,4132,202,20.455446,0.465347,6.924812,3,0,5,318,...,210,143,0,2,3,0,22,48,1,27
3,catullus.carmina1_63.txt,6361,395,15.959494,0.283544,5.720000,10,0,8,567,...,285,195,1,3,2,0,24,61,10,59
4,claudian.carmina_minora.txt,10828,642,16.852025,0.218069,6.248677,13,0,6,933,...,212,341,0,0,5,0,39,46,9,26


In [6]:
df_prose_data.head()

,File Name,word_count,sentence_count,sentence_length,fraction_sentence_relative,relative_clause_length,alius,antequam,atque_consonant,conjunction,...,personal,preposition,priusquam,quidam,quin,quominus,reflexive,si,superlative,ut
0,ammianus.rerum_gestarum.part.14.txt,8221,269,30.505576,0.605948,5.661017,27,1,11,719,...,60,527,0,47,1,0,38,31,19,108
1,ammianus.rerum_gestarum.part.15.txt,6872,247,27.740891,0.481781,5.647399,25,3,2,576,...,58,434,0,22,0,0,37,19,21,105
2,ammianus.rerum_gestarum.part.16.txt,7676,243,31.448560,0.547325,5.472222,25,2,2,673,...,79,470,0,40,1,0,54,27,24,95
3,ammianus.rerum_gestarum.part.17.txt,7396,266,27.755639,0.503759,5.323077,20,2,4,641,...,65,400,0,18,0,0,39,27,21,93
4,ammianus.rerum_gestarum.part.18.txt,5042,152,33.131579,0.578947,5.231343,11,0,2,423,...,50,318,0,16,2,0,16,17,33,55


In [7]:
# merge verse genre labels

df_verse = pd.merge(df_verse_labels, df_verse_data, on='File Name')
df_verse.to_csv('verse_labeled.csv')
df_verse.head()

,Author,Work,Genre,File Name,word_count,sentence_count,sentence_length,fraction_sentence_relative,relative_clause_length,alius,...,personal,preposition,priusquam,quidam,quin,quominus,reflexive,si,superlative,ut
0,Plautus,Amphitruo,drama,plautus.amphitruo.txt,9340,987,9.460993,0.198582,4.879668,17,...,738,326,3,2,21,0,44,123,3,139
1,Plautus,Asinaria,drama,plautus.asinaria.txt,7632,901,8.467259,0.178690,4.354167,19,...,622,244,4,0,23,0,37,130,5,105
2,Plautus,Aulularia,drama,plautus.aulularia.txt,6561,744,8.813172,0.186828,4.760479,9,...,582,210,4,0,18,0,24,89,5,89
3,Plautus,Bacchides,drama,plautus.bacchides.txt,9458,1182,7.996616,0.170051,4.745690,13,...,728,316,5,1,15,0,39,110,6,152
4,Plautus,Captivi,drama,plautus.captivi.txt,8288,819,10.117216,0.230769,4.714894,20,...,691,335,2,0,15,1,37,113,7,112


In [8]:
# merge prose genre labels

df_prose = pd.merge(df_prose_labels, df_prose_data, on='File Name')
df_prose.to_csv('prose_labeled.csv')
df_prose.head()

,Author,Work,Genre,File Name,word_count,sentence_count,sentence_length,fraction_sentence_relative,relative_clause_length,alius,...,personal,preposition,priusquam,quidam,quin,quominus,reflexive,si,superlative,ut
0,Augustine,Epistulae (1-10),epistolography,augustine.epistulae_1_10.part.1.txt,4791,270,17.722222,0.474074,5.107477,22,...,207,249,3,17,0,0,26,58,37,68
1,Augustine,Epistulae (11-20),epistolography,augustine.epistulae_11_20.part.2.txt,4815,195,24.682051,0.625641,5.363636,10,...,212,278,0,21,0,0,13,84,29,66
2,Augustine,Epistulae (21-30),epistolography,augustine.epistulae_21_30.part.3.txt,13013,476,27.342437,0.581933,6.191489,27,...,616,767,1,10,2,0,39,173,66,165
3,Augustine,Epistulae (31-40),epistolography,augustine.epistulae_31_40.part.4.txt,14078,615,22.868293,0.557724,5.590510,46,...,435,829,0,15,0,1,56,185,65,190
4,Augustine,Epistulae (41-50),epistolography,augustine.epistulae_41_50.part.5.txt,12367,456,27.114035,0.629386,5.595978,46,...,326,874,1,16,0,0,60,190,49,134


In [9]:
import pandas as pd
import numpy as np
from collections import Counter

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score
from sklearn.inspection import permutation_importance

def train_evaluate_genre_100times_with_misclassified(df: pd.DataFrame, 
                                                     target_genre: str,
                                                     base_random_state: int = 42,
                                                     n_estimators: int = 100):
    """
    Runs a one-vs-rest random forest classification 100 times, each time with 
    a different random seed for the 5-fold stratified cross-validation. 
    Reports mean/std for accuracy and F1 across all trials, and mean/std for 
    Gini and permutation importances across all trials.
    
    Also tracks how many times each text is misclassified across all 100 trials
    (500 folds total) and reports any text that is misclassified at least once.

    Parameters
    ----------
    df : pd.DataFrame
        DataFrame containing columns:
            - "Genre" (the class label)
            - "File Name" (document identity)
            - 26 numerical features in columns "sentence_length" to "ut".
    target_genre : str
        The genre to treat as the positive class in the one-vs-rest setting.
    base_random_state : int, optional
        Starting random state for reproducibility, by default 42.
        Trial i will use random_state = base_random_state + i
    n_estimators : int, optional
        Number of trees in the random forest, by default 100.

    Returns
    -------
    None
        Prints out:
            (1) Mean/stdev of accuracy and macro-F1 (across 100 trials,
                each trial does 5-fold CV internally)
            (2) Mean/stdev of Gini importance across 100 final models
            (3) Mean/stdev of permutation importance across 100 final models
            (4) A report of which texts (File Name) were misclassified at 
                least once and how many times total.
    """

    # Identify the feature columns
    feature_cols = df.loc[:, "sentence_length":"ut"].columns
    X = df[feature_cols].values
    
    # Convert Genre to binary label (one-vs-rest)
    y = (df["Genre"] == target_genre).astype(int).values
    
    # Also store file names for tracking misclassifications
    file_names = df["File Name"].values

    # For reproducibility, define 100 different seeds
    random_seeds = [base_random_state + i for i in range(100)]
    
    # Collect metrics and importances across the 100 trials
    trial_accuracies = []
    trial_f1s = []
    # Gini importances and Permutation importances are shape (100, n_features)
    gini_all = []
    perm_all = []
    
    # Counter to track total misclassifications across all folds and trials
    misclassified_counter = Counter()

    for seed in random_seeds:
        # =========================================================
        # 1) 5-fold Stratified CV with this random seed
        # =========================================================
        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed)

        fold_accuracies = []
        fold_f1s = []

        for train_idx, test_idx in skf.split(X, y):
            X_train, X_test = X[train_idx], X[test_idx]
            y_train, y_test = y[train_idx], y[test_idx]
            file_test = file_names[test_idx]

            # Train the random forest
            rf = RandomForestClassifier(n_estimators=n_estimators, 
                                        random_state=seed)
            rf.fit(X_train, y_train)

            # Predictions
            y_pred = rf.predict(X_test)
            fold_accuracies.append(accuracy_score(y_test, y_pred))
            fold_f1s.append(f1_score(y_test, y_pred, average="macro"))

            # ----------------------------------------------------
            # Track which texts are misclassified in this fold
            # ----------------------------------------------------
            mis_mask = (y_pred != y_test)
            mis_file_names = file_test[mis_mask]
            for f_name in mis_file_names:
                misclassified_counter[f_name] += 1
        
        # Mean metrics across the 5 folds for this trial
        trial_accuracies.append(np.mean(fold_accuracies))
        trial_f1s.append(np.mean(fold_f1s))

        # =========================================================
        # 2) Retrain on entire dataset for final feature importance
        #    (this is still the same "trial" seed)
        # =========================================================
        final_rf = RandomForestClassifier(n_estimators=n_estimators,
                                          random_state=seed)
        final_rf.fit(X, y)

        # Gini importances
        gini_importances = final_rf.feature_importances_

        # Permutation importances
        perm_result = permutation_importance(
            estimator=final_rf,
            X=X,
            y=y,
            n_repeats=20,
            random_state=seed
        )
        perm_importances = perm_result.importances_mean  # average over repeats

        gini_all.append(gini_importances)
        perm_all.append(perm_importances)
    
    # ----------------------------------------------------
    # Compute overall mean/std of accuracy & F1 across 100 trials
    # ----------------------------------------------------
    mean_acc_100 = np.mean(trial_accuracies)
    std_acc_100  = np.std(trial_accuracies)
    mean_f1_100  = np.mean(trial_f1s)
    std_f1_100   = np.std(trial_f1s)

    print("=== Results Over 100 Trials (Each w/ 5-fold CV) ===")
    print(f"Accuracy: {mean_acc_100:.3f} ± {std_acc_100:.3f}")
    print(f"Macro F1: {mean_f1_100:.3f} ± {std_f1_100:.3f}\n")
    
    # ----------------------------------------------------
    # Compute overall mean/std of Gini and Permutation importances
    # ----------------------------------------------------
    gini_all = np.array(gini_all)  # shape = (100, n_features)
    perm_all = np.array(perm_all)  # shape = (100, n_features)

    gini_mean = gini_all.mean(axis=0)
    gini_std  = gini_all.std(axis=0)

    perm_mean = perm_all.mean(axis=0)
    perm_std  = perm_all.std(axis=0)

    # Make a dataframe for Gini importances
    gini_df = pd.DataFrame({
        "Feature Name": feature_cols,
        "Mean Gini": gini_mean,
        "Std Gini": gini_std
    }).sort_values("Mean Gini", ascending=False)

    # Make a dataframe for Permutation importances
    perm_df = pd.DataFrame({
        "Feature Name": feature_cols,
        "Mean Permutation Importance": perm_mean,
        "Std Permutation Importance": perm_std
    }).sort_values("Mean Permutation Importance", ascending=False)

    print("=== Gini Importances (Mean ± Std over 100 Trials) ===")
    print(gini_df.to_string(index=False))
    print()

    print("=== Permutation Importances (Mean ± Std over 100 Trials) ===")
    print(perm_df.to_string(index=False))
    print()

    # ----------------------------------------------------
    # 3) Report misclassified texts (across all 100 trials)
    # ----------------------------------------------------
    misclass_list = [
        (fname, count) 
        for fname, count in misclassified_counter.items() 
        if count > 0
    ]

    # Sort by descending times misclassified
    misclass_list.sort(key=lambda x: x[1], reverse=True)

    if len(misclass_list) > 0:
        misclass_df = pd.DataFrame(misclass_list, 
                                   columns=["File Name", "Times Misclassified"])
        print("=== Misclassified Texts Over 100 Trials ===")
        print(misclass_df.to_string(index=False))
    else:
        print("No texts were misclassified in any fold of any trial!")


In [10]:
train_evaluate_genre_100times_with_misclassified(df_verse, 'drama', 42, 200)

=== Results Over 100 Trials (Each w/ 5-fold CV) ===
Accuracy: 0.965 ± 0.006
Macro F1: 0.927 ± 0.013

=== Gini Importances (Mean ± Std over 100 Trials) ===
              Feature Name  Mean Gini  Std Gini
             interrogative   0.209114  0.021365
           sentence_length   0.131830  0.016193
                      iste   0.129951  0.015328
                  personal   0.071309  0.012415
             demonstrative   0.068226  0.013507
                      quin   0.045898  0.009562
fraction_sentence_relative   0.043623  0.006589
                        si   0.040844  0.009010
                 priusquam   0.035357  0.009188
                        ut   0.025687  0.006133
                      idem   0.021337  0.005127
                 gerundive   0.021236  0.002772
               conjunction   0.018539  0.002853
               superlative   0.018443  0.002451
                cum_clause   0.017069  0.003022
                  antequam   0.014629  0.002260
                      ipse   

In [11]:
train_evaluate_genre_100times_with_misclassified(df_verse, 'epic', 42, 200)

=== Results Over 100 Trials (Each w/ 5-fold CV) ===
Accuracy: 0.864 ± 0.012
Macro F1: 0.862 ± 0.012

=== Gini Importances (Mean ± Std over 100 Trials) ===
              Feature Name  Mean Gini  Std Gini
                  personal   0.152267  0.009387
           sentence_length   0.092796  0.006739
    relative_clause_length   0.080509  0.006903
                      iste   0.076259  0.007751
             interrogative   0.068481  0.006118
               conjunction   0.061965  0.004656
                        si   0.061747  0.006501
               preposition   0.060961  0.003765
             demonstrative   0.050171  0.004914
                 reflexive   0.041791  0.002946
                        ut   0.038195  0.004026
                cum_clause   0.026558  0.002900
fraction_sentence_relative   0.021470  0.002333
            o_interjection   0.020542  0.001941
                 gerundive   0.020380  0.002030
                      idem   0.019844  0.002538
                      ipse   

In [12]:
train_evaluate_genre_100times_with_misclassified(df_verse, 'elegy', 42, 200)

=== Results Over 100 Trials (Each w/ 5-fold CV) ===
Accuracy: 0.927 ± 0.006
Macro F1: 0.782 ± 0.025

=== Gini Importances (Mean ± Std over 100 Trials) ===
              Feature Name  Mean Gini  Std Gini
    relative_clause_length   0.111411  0.007020
               conjunction   0.077678  0.007173
                        ut   0.067875  0.005928
                 gerundive   0.067790  0.005575
             demonstrative   0.062519  0.005720
                 reflexive   0.057226  0.005946
                      ipse   0.050069  0.004943
                  personal   0.049052  0.004364
           sentence_length   0.043343  0.005002
fraction_sentence_relative   0.043325  0.004867
           atque_consonant   0.043320  0.004709
               preposition   0.038152  0.004721
                    quidam   0.034516  0.003667
             interrogative   0.034419  0.004346
                     alius   0.033852  0.004415
                        si   0.028320  0.003605
                      idem   

In [13]:
train_evaluate_genre_100times_with_misclassified(df_verse, 'miscellaneous', 42, 200)

=== Results Over 100 Trials (Each w/ 5-fold CV) ===
Accuracy: 0.861 ± 0.010
Macro F1: 0.713 ± 0.018

=== Gini Importances (Mean ± Std over 100 Trials) ===
              Feature Name  Mean Gini  Std Gini
               preposition   0.138867  0.009573
               conjunction   0.084313  0.006914
                 reflexive   0.072935  0.005595
                  personal   0.063854  0.005177
                 gerundive   0.058699  0.005466
                      ipse   0.045115  0.003870
             demonstrative   0.044264  0.004481
           sentence_length   0.044095  0.003574
                        ut   0.043936  0.003843
            o_interjection   0.042450  0.003017
                        si   0.040374  0.003516
    relative_clause_length   0.037758  0.003085
             interrogative   0.035948  0.002979
           atque_consonant   0.035945  0.002984
                cum_clause   0.033714  0.002861
fraction_sentence_relative   0.032323  0.003391
                     alius   

In [14]:
train_evaluate_genre_100times_with_misclassified(df_prose, 'epistolography', 42, 200)

=== Results Over 100 Trials (Each w/ 5-fold CV) ===
Accuracy: 0.944 ± 0.005
Macro F1: 0.892 ± 0.010

=== Gini Importances (Mean ± Std over 100 Trials) ===
              Feature Name  Mean Gini  Std Gini
                  personal   0.163909  0.008498
           sentence_length   0.124557  0.007418
             interrogative   0.055662  0.005574
           atque_consonant   0.052934  0.004699
                 reflexive   0.052344  0.005876
fraction_sentence_relative   0.047067  0.004248
                cum_clause   0.044250  0.004242
                        si   0.041705  0.004108
                        ut   0.041172  0.003977
                      idem   0.040547  0.003975
                      iste   0.036881  0.002944
             demonstrative   0.034203  0.004097
                     alius   0.031687  0.003493
    relative_clause_length   0.029315  0.002852
                 gerundive   0.026005  0.003647
               conjunction   0.025819  0.003257
               superlative   

In [15]:
train_evaluate_genre_100times_with_misclassified(df_prose, 'historiography', 42, 200)

=== Results Over 100 Trials (Each w/ 5-fold CV) ===
Accuracy: 0.949 ± 0.006
Macro F1: 0.924 ± 0.009

=== Gini Importances (Mean ± Std over 100 Trials) ===
              Feature Name  Mean Gini  Std Gini
                      iste   0.181526  0.010314
                 reflexive   0.090939  0.005367
             interrogative   0.080730  0.006407
                        si   0.080057  0.005009
           sentence_length   0.050808  0.004335
                  personal   0.046389  0.004050
                cum_clause   0.045590  0.003151
             demonstrative   0.042508  0.003604
                       dum   0.039806  0.003368
                        ut   0.033907  0.003362
               preposition   0.030735  0.003610
                      idem   0.028286  0.002834
    relative_clause_length   0.027863  0.002530
                 gerundive   0.025424  0.002337
                      ipse   0.020742  0.002379
                 priusquam   0.020302  0.002210
               conjunction   

In [16]:
train_evaluate_genre_100times_with_misclassified(df_prose, 'oratory', 42, 200)

=== Results Over 100 Trials (Each w/ 5-fold CV) ===
Accuracy: 0.922 ± 0.006
Macro F1: 0.843 ± 0.015

=== Gini Importances (Mean ± Std over 100 Trials) ===
              Feature Name  Mean Gini  Std Gini
             interrogative   0.143778  0.007726
               conjunction   0.092479  0.006489
                     alius   0.076899  0.005477
                      iste   0.063263  0.004900
                    quidam   0.054355  0.004607
    relative_clause_length   0.052347  0.003982
                       dum   0.049976  0.004573
               superlative   0.048082  0.003824
               preposition   0.040846  0.004110
           sentence_length   0.038069  0.002977
            o_interjection   0.036244  0.003248
fraction_sentence_relative   0.033408  0.002821
                  personal   0.031627  0.002436
                        si   0.028116  0.002775
                 gerundive   0.026220  0.002548
             demonstrative   0.025730  0.002466
                cum_clause   

In [17]:
train_evaluate_genre_100times_with_misclassified(df_prose, 'philosophy', 42, 200)

=== Results Over 100 Trials (Each w/ 5-fold CV) ===
Accuracy: 0.903 ± 0.007
Macro F1: 0.737 ± 0.025

=== Gini Importances (Mean ± Std over 100 Trials) ===
              Feature Name  Mean Gini  Std Gini
    relative_clause_length   0.083974  0.004251
             interrogative   0.081337  0.004440
fraction_sentence_relative   0.077170  0.004307
               superlative   0.072312  0.004115
           sentence_length   0.058933  0.004556
                      iste   0.053471  0.003634
                  personal   0.044359  0.003405
                     alius   0.044254  0.003531
               preposition   0.043123  0.003720
                        si   0.042337  0.003349
                 reflexive   0.040922  0.003672
                    quidam   0.040204  0.003123
                      ipse   0.032704  0.002551
               conjunction   0.031982  0.002606
             demonstrative   0.031967  0.003053
                      idem   0.027687  0.002352
                        ut   

In [18]:
train_evaluate_genre_100times_with_misclassified(df_prose, 'technical treatise', 42, 200)

=== Results Over 100 Trials (Each w/ 5-fold CV) ===
Accuracy: 0.963 ± 0.005
Macro F1: 0.897 ± 0.019

=== Gini Importances (Mean ± Std over 100 Trials) ===
              Feature Name  Mean Gini  Std Gini
                  personal   0.149879  0.008941
                 gerundive   0.082896  0.006087
                        si   0.060441  0.005994
               conjunction   0.060068  0.005707
                 reflexive   0.054993  0.005483
                      iste   0.054660  0.005681
             interrogative   0.052140  0.004723
               preposition   0.049820  0.005555
                cum_clause   0.045113  0.004864
                    quidam   0.044720  0.004056
    relative_clause_length   0.041487  0.004386
                      ipse   0.036077  0.004435
             demonstrative   0.035492  0.004575
                      idem   0.029543  0.003431
           sentence_length   0.026071  0.003388
                     alius   0.025265  0.003484
                        ut   

In [19]:
train_evaluate_genre_100times_with_misclassified(df_prose, 'miscellaneous', 42, 200)

=== Results Over 100 Trials (Each w/ 5-fold CV) ===
Accuracy: 0.956 ± 0.005
Macro F1: 0.885 ± 0.016

=== Gini Importances (Mean ± Std over 100 Trials) ===
              Feature Name  Mean Gini  Std Gini
           atque_consonant   0.117209  0.008351
                      iste   0.098351  0.007299
                        ut   0.072846  0.006097
                        si   0.063969  0.004860
             interrogative   0.062790  0.006057
                    quidam   0.052844  0.004969
                      idem   0.046878  0.004533
                  personal   0.044159  0.004192
               conjunction   0.040026  0.003773
fraction_sentence_relative   0.036938  0.003394
               superlative   0.034955  0.003226
           sentence_length   0.033385  0.003470
                     alius   0.030348  0.002913
                      ipse   0.029256  0.003403
               preposition   0.027934  0.003342
    relative_clause_length   0.025747  0.002749
                 gerundive   

In [47]:
def repeated_one_vs_rest_classification(
    df: pd.DataFrame,
    target_genre: str,
    n_repeats: int = 5,
    n_estimators: int = 100,
    random_state: int = 42
):
    """
    Perform one-vs.-rest classification for a given genre using a Random Forest.
    
    Parameters
    ----------
    df : pd.DataFrame
        DataFrame containing columns:
            - "Genre": the class label.
            - "File Name": the unique identifier for each text/document.
            - Numerical feature columns from "sentence_length" to "ut" (26 columns).
    target_genre : str
        The genre for which we do one-vs.-rest classification.
    n_repeats : int, optional
        Number of times to repeat the entire 5-fold stratified cross-validation. 
        (Default is 5)
    n_estimators : int, optional
        Number of trees in the random forest. (Default is 100)
    random_state : int, optional
        Random seed for reproducibility. (Default is 42)
    
    Returns
    -------
    results : dict
        Dictionary containing the following keys:
            - "accuracy_mean": float
            - "accuracy_std": float
            - "f1_mean": float
            - "f1_std": float
            - "misclass_fraction_df": pd.DataFrame 
                columns = ["File Name", "Fraction of Misclassifications"]
            - "gini_importance_df": pd.DataFrame 
                columns = ["Feature Name", "Mean Gini", "SD Gini"]
            - "perm_importance_df": pd.DataFrame 
                columns = ["Feature Name", "Mean Permutation Importance", "SD Permutation Importance"]
    """
    
    # -----------------------------
    # 1) Prepare data
    # -----------------------------
    # Identify feature columns (assuming they are from "sentence_length" to "ut")
    # If you prefer an explicit list, you can replace this part with a known list of columns.
    feature_cols = df.loc[:, "sentence_length":"ut"].columns.tolist()
    
    # Binarize the target: 1 if Genre == target_genre, else 0
    df = df.copy()  # To avoid modifying the original DataFrame
    df["target"] = (df["Genre"] == target_genre).astype(int)
    
    X = df[feature_cols].values
    y = df["target"].values
    file_names = df["File Name"].values
    
    # -----------------------------
    # 2) Set up repeated stratified 5-fold cross-validation
    # -----------------------------
    rskf = RepeatedStratifiedKFold(
        n_splits=5, 
        n_repeats=n_repeats, 
        random_state=random_state
    )
    
    # -----------------------------
    # 3) Storage for metrics and importances
    # -----------------------------
    accuracies = []
    f1_scores = []
    
    # Track misclassifications per document
    # key: file_name -> [times_in_test_set, times_misclassified]
    misclass_dict = {}
    for fn in file_names:
        misclass_dict[fn] = [0, 0]

    # For storing Gini importances: one list per fold
    gini_importances_list = []
    # For storing Permutation importances: one list per fold
    perm_importances_list = []
    
    # -----------------------------
    # 4) Cross-validation loop
    # -----------------------------
    for train_idx, test_idx in rskf.split(X, y):
        # Split data
        X_train, X_test = X[train_idx], X[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]
        fn_test = file_names[test_idx]
        
        # Random Forest classifier
        rf = RandomForestClassifier(
            n_estimators=n_estimators, 
            random_state=random_state
        )
        rf.fit(X_train, y_train)
        
        # Predictions
        y_pred = rf.predict(X_test)
        
        # Accuracy, Macro F1
        acc = accuracy_score(y_test, y_pred)
        f1 = f1_score(y_test, y_pred, average='macro')
        
        accuracies.append(acc)
        f1_scores.append(f1)
        
        # Track misclassifications for each doc
        for i, pred in enumerate(y_pred):
            true_label = y_test[i]
            misclass_dict[fn_test[i]][0] += 1  # tested once more
            if pred != true_label:
                misclass_dict[fn_test[i]][1] += 1
        
        # Gini feature importances
        gini_importances_list.append(rf.feature_importances_)
        
        # Permutation importance
        perm_imp = permutation_importance(
            rf, X_test, y_test, 
            n_repeats=10, 
            random_state=random_state, 
            n_jobs=-1
        )
        perm_importances_list.append(perm_imp.importances_mean)
    
    # -----------------------------
    # 5) Aggregate results
    # -----------------------------
    # Accuracy and F1
    accuracy_mean = np.mean(accuracies)
    accuracy_std = np.std(accuracies)
    f1_mean = np.mean(f1_scores)
    f1_std = np.std(f1_scores)
    
    # -----------------------------
    # 6) Fraction of times each doc is misclassified
    # -----------------------------
    misclass_fractions = []
    for fn, (times_tested, times_miscl) in misclass_dict.items():
        frac_miscl = times_miscl / times_tested if times_tested > 0 else 0.0
        misclass_fractions.append((fn, frac_miscl))
    misclass_fraction_df = pd.DataFrame(
        misclass_fractions, 
        columns=["File Name", "Fraction of Misclassifications"]
    )
    
    # -----------------------------
    # 7) Mean and std of Gini importances
    # -----------------------------
    gini_array = np.array(gini_importances_list)  # shape: (n_folds_total, n_features)
    gini_means = np.mean(gini_array, axis=0)
    gini_stds = np.std(gini_array, axis=0)
    
    gini_importance_df = pd.DataFrame({
        "Feature Name": feature_cols,
        "Mean Gini": gini_means,
        "SD Gini": gini_stds
    }).sort_values("Mean Gini", ascending=False).reset_index(drop=True)
    
    # -----------------------------
    # 8) Mean and std of permutation importances
    # -----------------------------
    perm_array = np.array(perm_importances_list)  # shape: (n_folds_total, n_features)
    perm_means = np.mean(perm_array, axis=0)
    perm_stds = np.std(perm_array, axis=0)
    
    perm_importance_df = pd.DataFrame({
        "Feature Name": feature_cols,
        "Mean Permutation Importance": perm_means,
        "SD Permutation Importance": perm_stds
    }).sort_values("Mean Permutation Importance", ascending=False).reset_index(drop=True)
    
    # -----------------------------
    # 9) Prepare final output
    # -----------------------------
    results = {
        "accuracy_mean": accuracy_mean,
        "accuracy_std": accuracy_std,
        "f1_mean": f1_mean,
        "f1_std": f1_std,
        "misclass_fraction_df": misclass_fraction_df,
        "gini_importance_df": gini_importance_df,
        "perm_importance_df": perm_importance_df
    }
    
    return results

In [48]:
# Example usage:
df = df_verse
results = repeated_one_vs_rest_classification(df, target_genre="Epic", n_repeats=5)

# # Access results
print("Mean accuracy:", results["accuracy_mean"])
print("Std accuracy:", results["accuracy_std"])
print("Mean F1 (macro):", results["f1_mean"])
print("Std F1 (macro):", results["f1_std"])
# 
# # Misclassification fraction per document
print(results["misclass_fraction_df"])
# 
# # Feature importances
print(results["gini_importance_df"])
print(results["perm_importance_df"])

Mean accuracy: 1.0
Std accuracy: 0.0
Mean F1 (macro): 1.0
Std F1 (macro): 0.0
                     File Name  Fraction of Misclassifications
0        plautus.amphitruo.txt                             0.0
1         plautus.asinaria.txt                             0.0
2        plautus.aulularia.txt                             0.0
3        plautus.bacchides.txt                             0.0
4          plautus.captivi.txt                             0.0
..                         ...                             ...
229  statius.silvae.part.2.txt                             0.0
230  statius.silvae.part.3.txt                             0.0
231  statius.silvae.part.4.txt                             0.0
232  statius.silvae.part.5.txt                             0.0
233        vergil.eclogues.txt                             0.0

[234 rows x 2 columns]
                  Feature Name  Mean Gini  SD Gini
0              sentence_length        0.0      0.0
1   fraction_sentence_relative        0.

In [13]:
# epic dataset

df_epic = df_verse.loc[(df_verse['Genre'] == 'epic')]
df_epic = df_epic.loc[df_epic['Author'].isin(['Catullus', 'Ovid', 'Lucan', 'Statius', 'Valerius Flaccus', 'Vergil'])]

df_silius = df_verse.loc[(df_verse['Author'] == 'Silius Italicus')]
df_manilius = df_verse.loc[(df_verse['Author'] == 'Manilius')]
df_lucretius = df_verse.loc[(df_verse['Author'] == 'Lucretius')]

df_epic = pd.concat([df_silius, df_manilius, df_lucretius, df_epic])
df_epic.to_csv('stylometry_data_epic.csv')
df_epic

,Author,Work,Genre,File Name,word_count,sentence_count,sentence_length,fraction_sentence_relative,relative_clause_length,alius,...,personal,preposition,priusquam,quidam,quin,quominus,reflexive,si,superlative,ut


In [14]:
# satire dataset

df_epic = df_verse.loc[(df_verse['Genre'] == 'epic')]
df_epic = df_epic.loc[df_epic['Author'].isin(['Catullus', 'Vergil', 'Ovid', 'Lucan', 'Silius Italicus', 'Statius', 'Valerius Flaccus'])]
df_epic = df_epic[df_epic['Work'] != 'Georgics']
df_satire = df_verse.loc[(df_verse['Work'] == 'Satires')]
df_epic_satire = pd.concat([df_epic,df_satire])

df_epic_satire.to_csv('stylometry_data_satire.csv')
df_epic_satire

,Author,Work,Genre,File Name,word_count,sentence_count,sentence_length,fraction_sentence_relative,relative_clause_length,alius,...,personal,preposition,priusquam,quidam,quin,quominus,reflexive,si,superlative,ut


In [76]:
# custom sorting function
def sort_by_first_and_third_word(df, column_name):
    """Sorts a DataFrame alphabetically by the first and third word in a column."""

    df['first_word'] = df[column_name].str.split().str[0]
    df['third_word'] = df[column_name].str.split().str[2]
    df.sort_values(by=['first_word', 'third_word'], inplace=True)
    df.drop(['first_word', 'third_word'], axis=1, inplace=True)
    return df

# Example usage:
df = pd.DataFrame({'text_column': ['the quick brown fox', 'jumps over the lazy dog', 'a quick brown fox']})
df = sort_by_first_and_third_word(df, 'text_column')
#print(df)

In [80]:
# mortal and divine speech

df_mortal_divine = pd.read_csv('20241125_053108-normed.csv')
df_mortal_divine = df_mortal_divine.rename(columns={'Unnamed: 0': 'File Name'}) 
df_mortal_divine = df_mortal_divine.sort_values(by='File Name')
df_mortal_divine = sort_by_first_and_third_word(df_mortal_divine, 'File Name')
df_mortal_divine_base = df_mortal_divine[df_mortal_divine['File Name'] != 'aeneid 12 divine speech no juturna.txt']
df_mortal_divine_base.to_csv('mortal_divine_base_speech.csv')
df_mortal_divine_base

,File Name,word_count,sentence_count,sentence_length,fraction_sentence_relative,relative_clause_length,alius,antequam,atque_consonant,conjunction,...,personal,preposition,priusquam,quidam,quin,quominus,reflexive,si,superlative,ut
0,aeneid 1 divine speech.txt,1414,64,19.234375,0.328125,5.560000,0.000707,0.0,0.000000,0.089109,...,0.027581,0.027581,0.000000,0.0,0.000707,0.0,0.002122,0.001414,0.001414,0.002829
2,aeneid 10 divine speech.txt,1142,70,14.228571,0.142857,4.538462,0.000876,0.0,0.001751,0.095447,...,0.017513,0.024518,0.000000,0.0,0.002627,0.0,0.000876,0.008757,0.000876,0.003503
4,aeneid 11 divine speech.txt,539,27,17.333333,0.074074,5.666667,0.001855,0.0,0.000000,0.068646,...,0.014842,0.044527,0.000000,0.0,0.000000,0.0,0.007421,0.000000,0.000000,0.003711
7,aeneid 12 divine speech.txt,709,49,12.653061,0.183673,4.545455,0.001410,0.0,0.000000,0.084626,...,0.033850,0.033850,0.000000,0.0,0.000000,0.0,0.001410,0.004231,0.001410,0.005642
11,aeneid 4 divine speech.txt,554,37,13.027027,0.216216,5.625000,0.000000,0.0,0.000000,0.108303,...,0.034296,0.021661,0.000000,0.0,0.001805,0.0,0.000000,0.012635,0.000000,0.000000
15,aeneid 7 divine speech.txt,626,35,15.628571,0.200000,6.300000,0.000000,0.0,0.004792,0.087859,...,0.031949,0.035144,0.000000,0.0,0.003195,0.0,0.000000,0.007987,0.000000,0.001597
17,aeneid 8 divine speech.txt,465,26,15.576923,0.307692,6.000000,0.002151,0.0,0.000000,0.073118,...,0.038710,0.025806,0.000000,0.0,0.000000,0.0,0.000000,0.004301,0.004301,0.002151
1,aeneid 1 mortal speech.txt,1178,65,15.846154,0.215385,4.562500,0.000000,0.0,0.000849,0.094228,...,0.029711,0.032258,0.000000,0.0,0.000000,0.0,0.001698,0.010187,0.000849,0.000849
3,aeneid 10 mortal speech.txt,1012,79,11.151899,0.202532,4.647059,0.000000,0.0,0.000000,0.074111,...,0.041502,0.028656,0.000000,0.0,0.000000,0.0,0.000988,0.003953,0.002964,0.000988
5,aeneid 11 mortal speech.txt,2628,145,15.827586,0.275862,5.936170,0.001903,0.0,0.000000,0.095129,...,0.028539,0.031963,0.000000,0.0,0.001142,0.0,0.003044,0.010274,0.001142,0.002664


In [81]:
df_mortal_divine_no_juturna = df_mortal_divine[df_mortal_divine['File Name'] != 'aeneid 12 divine speech.txt']
df_mortal_divine_no_juturna.to_csv('mortal_divine_no_juturna_speech.csv')
df_mortal_divine_no_juturna

,File Name,word_count,sentence_count,sentence_length,fraction_sentence_relative,relative_clause_length,alius,antequam,atque_consonant,conjunction,...,personal,preposition,priusquam,quidam,quin,quominus,reflexive,si,superlative,ut
0,aeneid 1 divine speech.txt,1414,64,19.234375,0.328125,5.560000,0.000707,0.0,0.000000,0.089109,...,0.027581,0.027581,0.000000,0.0,0.000707,0.0,0.002122,0.001414,0.001414,0.002829
2,aeneid 10 divine speech.txt,1142,70,14.228571,0.142857,4.538462,0.000876,0.0,0.001751,0.095447,...,0.017513,0.024518,0.000000,0.0,0.002627,0.0,0.000876,0.008757,0.000876,0.003503
4,aeneid 11 divine speech.txt,539,27,17.333333,0.074074,5.666667,0.001855,0.0,0.000000,0.068646,...,0.014842,0.044527,0.000000,0.0,0.000000,0.0,0.007421,0.000000,0.000000,0.003711
6,aeneid 12 divine speech no juturna.txt,486,29,14.655172,0.206897,4.166667,0.000000,0.0,0.000000,0.092593,...,0.026749,0.034979,0.000000,0.0,0.000000,0.0,0.000000,0.004115,0.002058,0.008230
11,aeneid 4 divine speech.txt,554,37,13.027027,0.216216,5.625000,0.000000,0.0,0.000000,0.108303,...,0.034296,0.021661,0.000000,0.0,0.001805,0.0,0.000000,0.012635,0.000000,0.000000
15,aeneid 7 divine speech.txt,626,35,15.628571,0.200000,6.300000,0.000000,0.0,0.004792,0.087859,...,0.031949,0.035144,0.000000,0.0,0.003195,0.0,0.000000,0.007987,0.000000,0.001597
17,aeneid 8 divine speech.txt,465,26,15.576923,0.307692,6.000000,0.002151,0.0,0.000000,0.073118,...,0.038710,0.025806,0.000000,0.0,0.000000,0.0,0.000000,0.004301,0.004301,0.002151
1,aeneid 1 mortal speech.txt,1178,65,15.846154,0.215385,4.562500,0.000000,0.0,0.000849,0.094228,...,0.029711,0.032258,0.000000,0.0,0.000000,0.0,0.001698,0.010187,0.000849,0.000849
3,aeneid 10 mortal speech.txt,1012,79,11.151899,0.202532,4.647059,0.000000,0.0,0.000000,0.074111,...,0.041502,0.028656,0.000000,0.0,0.000000,0.0,0.000988,0.003953,0.002964,0.000988
5,aeneid 11 mortal speech.txt,2628,145,15.827586,0.275862,5.936170,0.001903,0.0,0.000000,0.095129,...,0.028539,0.031963,0.000000,0.0,0.001142,0.0,0.003044,0.010274,0.001142,0.002664


In [88]:
# variables to focus on
# age
# education
# chronotype
# literary experience

In [89]:
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

import os
import re
import pandas as pd
from sklearn.decomposition import PCA
#import matplotlib.pyplot as plt
import numpy as np

from datetime import datetime

In [90]:
# load text data

df_WWC = pd.read_csv('merged_dataset_v3.csv')
df_WWC.head()

/var/folders/qt/wx4bttqn4h13mk9smx_yjchc0000gn/T/ipykernel_81094/413186783.py:3: DtypeWarning: Columns (0,1) have mixed types. Specify dtype option on import or set low_memory=False.
  df_WWC = pd.read_csv('merged_dataset_v3.csv')


,IdIdioma_osv,Texto_osv_osv,filename,Core_universal_dependencies,Non-core_universal_dependencies,Nominal_universal_dependencies,Non-dependency_universal_dependencies,IdMundialTexto_granularidad,IdMundial_granularidad,Texto_granularidad_promedio_granularidad,...,Texto_tokenized_sa_num_syll_mediana_psyco,Texto_tokenized_sa_num_syll_curtosis_psyco,Texto_tokenized_sa_num_syll_skewness_psyco,Texto_tokenized_sa_NP_promedio_psyco,Texto_tokenized_sa_NP_minimo_psyco,Texto_tokenized_sa_NP_maximo_psyco,Texto_tokenized_sa_NP_std_psyco,Texto_tokenized_sa_NP_mediana_psyco,Texto_tokenized_sa_NP_curtosis_psyco,Texto_tokenized_sa_NP_skewness_psyco
0,1,0.019777141,22_153578.txt,136.0,203.0,350.0,238.0,126366.0,12.0,6.531250,...,3.0,-0.052469,0.489205,15.000000,0.0,96.0,18.381644,8.0,3.273263,1.925712
1,1,0.022520447,22_161477.txt,82.0,92.0,199.0,168.0,134265.0,12.0,7.236364,...,3.0,0.759256,-0.114001,8.941176,0.0,76.0,17.058621,6.0,11.062845,3.539034
2,1,0.016744167,22_169507.txt,94.0,154.0,269.0,186.0,142295.0,12.0,6.291080,...,3.0,0.243429,0.380496,12.875000,0.0,111.0,17.310927,6.0,8.443541,2.725065
3,1,0.017703811,22_177311.txt,87.0,127.0,237.0,138.0,150099.0,12.0,6.318750,...,3.0,-0.055742,0.508188,14.035088,0.0,89.0,18.048390,7.0,3.666035,2.002149
4,1,0.018634681,22_183429.txt,617.0,870.0,1854.0,1108.0,156217.0,12.0,6.384232,...,3.0,0.107460,0.419077,11.509477,0.0,111.0,15.372544,6.0,8.368224,2.676734


In [91]:
df_WWC

,IdIdioma_osv,Texto_osv_osv,filename,Core_universal_dependencies,Non-core_universal_dependencies,Nominal_universal_dependencies,Non-dependency_universal_dependencies,IdMundialTexto_granularidad,IdMundial_granularidad,Texto_granularidad_promedio_granularidad,...,Texto_tokenized_sa_num_syll_mediana_psyco,Texto_tokenized_sa_num_syll_curtosis_psyco,Texto_tokenized_sa_num_syll_skewness_psyco,Texto_tokenized_sa_NP_promedio_psyco,Texto_tokenized_sa_NP_minimo_psyco,Texto_tokenized_sa_NP_maximo_psyco,Texto_tokenized_sa_NP_std_psyco,Texto_tokenized_sa_NP_mediana_psyco,Texto_tokenized_sa_NP_curtosis_psyco,Texto_tokenized_sa_NP_skewness_psyco
0,1,0.019777141,22_153578.txt,136.0,203.0,350.0,238.0,126366.0,12.0,6.531250,...,3.0,-0.052469,0.489205,15.000000,0.0,96.0,18.381644,8.0,3.273263,1.925712
1,1,0.022520447,22_161477.txt,82.0,92.0,199.0,168.0,134265.0,12.0,7.236364,...,3.0,0.759256,-0.114001,8.941176,0.0,76.0,17.058621,6.0,11.062845,3.539034
2,1,0.016744167,22_169507.txt,94.0,154.0,269.0,186.0,142295.0,12.0,6.291080,...,3.0,0.243429,0.380496,12.875000,0.0,111.0,17.310927,6.0,8.443541,2.725065
3,1,0.017703811,22_177311.txt,87.0,127.0,237.0,138.0,150099.0,12.0,6.318750,...,3.0,-0.055742,0.508188,14.035088,0.0,89.0,18.048390,7.0,3.666035,2.002149
4,1,0.018634681,22_183429.txt,617.0,870.0,1854.0,1108.0,156217.0,12.0,6.384232,...,3.0,0.107460,0.419077,11.509477,0.0,111.0,15.372544,6.0,8.368224,2.676734
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
99184,NaN,NaN,97250_255155.txt,66.0,23.0,155.0,1330.0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
99185,NaN,NaN,97250_255155.txt,66.0,23.0,155.0,1330.0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
99186,NaN,NaN,97250_255155.txt,66.0,23.0,155.0,1330.0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
99187,NaN,NaN,97250_255155.txt,66.0,23.0,155.0,1330.0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [92]:
# load demographic data

df_demo = pd.read_csv('participant_data_formatted.csv')
df_demo.head()

,Unnamed: 0,idUsuario,Respuesta_Pregunta_1,Respuesta_Pregunta_2,Respuesta_Pregunta_3,Respuesta_Pregunta_4,Respuesta_Pregunta_5,Respuesta_Pregunta_6,FechaNacimiento,Genero,...,IdUsuarioCategoria,IdUsuarioSubCategoria,IdIdioma,FechaAlta,MiembroEquipoMundial,MiembroEquipoMundialOrden,MiembroEquipoTarea,IdPais_Nacimiento,IdIdioma_Materno,EducacionFormalYYYY
0,0,user_ID,Response_to_Question_1,Response_to_Question_2,Response_to_Question_3,Response_to_Question_4,Response_to_Question_5,Response_to_Question_6,Date_of_Birth,Gender,...,User_Category_ID,User_Subcategory_ID,Language_ID,Registration_Date,Global_Team_Member,Global_Team_Member_Order,Team_Task_Member,Country_of_Birth_ID,Native_Language_ID,Formal_Education_Year
1,1,22,30,1,4,1,2,1,25-03-1971,2,...,3,5,1,04:32.9,TRUE,7,Desarrollo y programaci√≥n,307,1,8
2,2,49504,33,4,NaN,NaN,NaN,NaN,11/9/74,1,...,3,5,1,31:08.2,TRUE,3,Producci√≥n y coordinaci√≥n general,307,1,22
3,3,49505,20,2,NaN,3,1,5,25-12-1979,1,...,3,5,1,55:32.2,TRUE,5,Equipo Todo Terreno,307,1,42
4,4,49506,24,NaN,NaN,4,NaN,4,3/6/96,1,...,3,5,1,02:24.5,TRUE,6,Comunicaci√≥n,307,1,22


In [93]:
df_demo

,Unnamed: 0,idUsuario,Respuesta_Pregunta_1,Respuesta_Pregunta_2,Respuesta_Pregunta_3,Respuesta_Pregunta_4,Respuesta_Pregunta_5,Respuesta_Pregunta_6,FechaNacimiento,Genero,...,IdUsuarioCategoria,IdUsuarioSubCategoria,IdIdioma,FechaAlta,MiembroEquipoMundial,MiembroEquipoMundialOrden,MiembroEquipoTarea,IdPais_Nacimiento,IdIdioma_Materno,EducacionFormalYYYY
0,0,user_ID,Response_to_Question_1,Response_to_Question_2,Response_to_Question_3,Response_to_Question_4,Response_to_Question_5,Response_to_Question_6,Date_of_Birth,Gender,...,User_Category_ID,User_Subcategory_ID,Language_ID,Registration_Date,Global_Team_Member,Global_Team_Member_Order,Team_Task_Member,Country_of_Birth_ID,Native_Language_ID,Formal_Education_Year
1,1,22,30,1,4,1,2,1,25-03-1971,2,...,3,5,1,04:32.9,TRUE,7,Desarrollo y programaci√≥n,307,1,8
2,2,49504,33,4,NaN,NaN,NaN,NaN,11/9/74,1,...,3,5,1,31:08.2,TRUE,3,Producci√≥n y coordinaci√≥n general,307,1,22
3,3,49505,20,2,NaN,3,1,5,25-12-1979,1,...,3,5,1,55:32.2,TRUE,5,Equipo Todo Terreno,307,1,42
4,4,49506,24,NaN,NaN,4,NaN,4,3/6/96,1,...,3,5,1,02:24.5,TRUE,6,Comunicaci√≥n,307,1,22
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8769,8769,81091,NaN,2,NaN,NaN,NaN,NaN,19-06-1996,1,...,3,5,1,44:53.2,FALSE,0,NaN,307,1,18
8770,8770,81092,16,NaN,NaN,NaN,NaN,NaN,1/9/92,3,...,3,5,1,46:42.6,FALSE,0,NaN,418,1,20
8771,8771,81093,22,3,8,4,7,7,30-12-1988,1,...,3,5,1,48:59.6,FALSE,0,NaN,418,1,20
8772,8772,81094,21,2,NaN,NaN,NaN,NaN,11/2/01,2,...,3,5,1,50:17.0,FALSE,0,NaN,339,1,14


In [94]:
# extract user IDs from filenames

df_WWC['idUsuario'] = df_WWC['filename'].str.split('_').str[0]

In [95]:
df_WWC.head()

,IdIdioma_osv,Texto_osv_osv,filename,Core_universal_dependencies,Non-core_universal_dependencies,Nominal_universal_dependencies,Non-dependency_universal_dependencies,IdMundialTexto_granularidad,IdMundial_granularidad,Texto_granularidad_promedio_granularidad,...,Texto_tokenized_sa_num_syll_curtosis_psyco,Texto_tokenized_sa_num_syll_skewness_psyco,Texto_tokenized_sa_NP_promedio_psyco,Texto_tokenized_sa_NP_minimo_psyco,Texto_tokenized_sa_NP_maximo_psyco,Texto_tokenized_sa_NP_std_psyco,Texto_tokenized_sa_NP_mediana_psyco,Texto_tokenized_sa_NP_curtosis_psyco,Texto_tokenized_sa_NP_skewness_psyco,idUsuario
0,1,0.019777141,22_153578.txt,136.0,203.0,350.0,238.0,126366.0,12.0,6.531250,...,-0.052469,0.489205,15.000000,0.0,96.0,18.381644,8.0,3.273263,1.925712,22
1,1,0.022520447,22_161477.txt,82.0,92.0,199.0,168.0,134265.0,12.0,7.236364,...,0.759256,-0.114001,8.941176,0.0,76.0,17.058621,6.0,11.062845,3.539034,22
2,1,0.016744167,22_169507.txt,94.0,154.0,269.0,186.0,142295.0,12.0,6.291080,...,0.243429,0.380496,12.875000,0.0,111.0,17.310927,6.0,8.443541,2.725065,22
3,1,0.017703811,22_177311.txt,87.0,127.0,237.0,138.0,150099.0,12.0,6.318750,...,-0.055742,0.508188,14.035088,0.0,89.0,18.048390,7.0,3.666035,2.002149,22
4,1,0.018634681,22_183429.txt,617.0,870.0,1854.0,1108.0,156217.0,12.0,6.384232,...,0.107460,0.419077,11.509477,0.0,111.0,15.372544,6.0,8.368224,2.676734,22


In [96]:
# merge datasets

df = pd.merge(df_WWC, df_demo, on="idUsuario")

In [97]:
df.head()

,IdIdioma_osv,Texto_osv_osv,filename,Core_universal_dependencies,Non-core_universal_dependencies,Nominal_universal_dependencies,Non-dependency_universal_dependencies,IdMundialTexto_granularidad,IdMundial_granularidad,Texto_granularidad_promedio_granularidad,...,IdUsuarioCategoria,IdUsuarioSubCategoria,IdIdioma,FechaAlta,MiembroEquipoMundial,MiembroEquipoMundialOrden,MiembroEquipoTarea,IdPais_Nacimiento,IdIdioma_Materno,EducacionFormalYYYY
0,1,0.019777141,22_153578.txt,136.0,203.0,350.0,238.0,126366.0,12.0,6.531250,...,3,5,1,04:32.9,TRUE,7,Desarrollo y programaci√≥n,307,1,8
1,1,0.022520447,22_161477.txt,82.0,92.0,199.0,168.0,134265.0,12.0,7.236364,...,3,5,1,04:32.9,TRUE,7,Desarrollo y programaci√≥n,307,1,8
2,1,0.016744167,22_169507.txt,94.0,154.0,269.0,186.0,142295.0,12.0,6.291080,...,3,5,1,04:32.9,TRUE,7,Desarrollo y programaci√≥n,307,1,8
3,1,0.017703811,22_177311.txt,87.0,127.0,237.0,138.0,150099.0,12.0,6.318750,...,3,5,1,04:32.9,TRUE,7,Desarrollo y programaci√≥n,307,1,8
4,1,0.018634681,22_183429.txt,617.0,870.0,1854.0,1108.0,156217.0,12.0,6.384232,...,3,5,1,04:32.9,TRUE,7,Desarrollo y programaci√≥n,307,1,8


In [98]:
df

,IdIdioma_osv,Texto_osv_osv,filename,Core_universal_dependencies,Non-core_universal_dependencies,Nominal_universal_dependencies,Non-dependency_universal_dependencies,IdMundialTexto_granularidad,IdMundial_granularidad,Texto_granularidad_promedio_granularidad,...,IdUsuarioCategoria,IdUsuarioSubCategoria,IdIdioma,FechaAlta,MiembroEquipoMundial,MiembroEquipoMundialOrden,MiembroEquipoTarea,IdPais_Nacimiento,IdIdioma_Materno,EducacionFormalYYYY
0,1,0.019777141,22_153578.txt,136.0,203.0,350.0,238.0,126366.0,12.0,6.531250,...,3,5,1,04:32.9,TRUE,7,Desarrollo y programaci√≥n,307,1,8
1,1,0.022520447,22_161477.txt,82.0,92.0,199.0,168.0,134265.0,12.0,7.236364,...,3,5,1,04:32.9,TRUE,7,Desarrollo y programaci√≥n,307,1,8
2,1,0.016744167,22_169507.txt,94.0,154.0,269.0,186.0,142295.0,12.0,6.291080,...,3,5,1,04:32.9,TRUE,7,Desarrollo y programaci√≥n,307,1,8
3,1,0.017703811,22_177311.txt,87.0,127.0,237.0,138.0,150099.0,12.0,6.318750,...,3,5,1,04:32.9,TRUE,7,Desarrollo y programaci√≥n,307,1,8
4,1,0.018634681,22_183429.txt,617.0,870.0,1854.0,1108.0,156217.0,12.0,6.384232,...,3,5,1,04:32.9,TRUE,7,Desarrollo y programaci√≥n,307,1,8
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
68621,NaN,NaN,80697_191733.txt,1.0,1.0,1.0,0.0,NaN,NaN,NaN,...,3,5,1,53:31.8,FALSE,0,NaN,307,1,19
68622,NaN,NaN,80723_160203.txt,0.0,0.0,0.0,0.0,NaN,NaN,NaN,...,3,5,1,11:42.8,FALSE,0,NaN,307,1,22
68623,NaN,NaN,80815_231919.txt,88.0,108.0,166.0,216.0,NaN,NaN,NaN,...,3,5,3,07:56.1,FALSE,0,NaN,324,3,25
68624,NaN,NaN,80815_255025.txt,89.0,63.0,224.0,283.0,NaN,NaN,NaN,...,3,5,3,07:56.1,FALSE,0,NaN,324,3,25


In [99]:
#delete sparse rows at end

df = df.drop(df.index[68326:68627])
df

,IdIdioma_osv,Texto_osv_osv,filename,Core_universal_dependencies,Non-core_universal_dependencies,Nominal_universal_dependencies,Non-dependency_universal_dependencies,IdMundialTexto_granularidad,IdMundial_granularidad,Texto_granularidad_promedio_granularidad,...,IdUsuarioCategoria,IdUsuarioSubCategoria,IdIdioma,FechaAlta,MiembroEquipoMundial,MiembroEquipoMundialOrden,MiembroEquipoTarea,IdPais_Nacimiento,IdIdioma_Materno,EducacionFormalYYYY
0,1,0.019777141,22_153578.txt,136.0,203.0,350.0,238.0,126366.0,12.0,6.531250,...,3,5,1,04:32.9,TRUE,7,Desarrollo y programaci√≥n,307,1,8
1,1,0.022520447,22_161477.txt,82.0,92.0,199.0,168.0,134265.0,12.0,7.236364,...,3,5,1,04:32.9,TRUE,7,Desarrollo y programaci√≥n,307,1,8
2,1,0.016744167,22_169507.txt,94.0,154.0,269.0,186.0,142295.0,12.0,6.291080,...,3,5,1,04:32.9,TRUE,7,Desarrollo y programaci√≥n,307,1,8
3,1,0.017703811,22_177311.txt,87.0,127.0,237.0,138.0,150099.0,12.0,6.318750,...,3,5,1,04:32.9,TRUE,7,Desarrollo y programaci√≥n,307,1,8
4,1,0.018634681,22_183429.txt,617.0,870.0,1854.0,1108.0,156217.0,12.0,6.384232,...,3,5,1,04:32.9,TRUE,7,Desarrollo y programaci√≥n,307,1,8
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
68321,1,0.009633,81093_268602.txt,18.0,20.0,39.0,42.0,232390.0,13.0,7.000000,...,3,5,1,48:59.6,FALSE,0,NaN,418,1,20
68322,1,0.017454,81093_271453.txt,14.0,19.0,51.0,25.0,235241.0,13.0,7.222222,...,3,5,1,48:59.6,FALSE,0,NaN,418,1,20
68323,1,0.019284,81093_274110.txt,15.0,21.0,53.0,22.0,237898.0,13.0,6.709677,...,3,5,1,48:59.6,FALSE,0,NaN,418,1,20
68324,1,0.029317,81093_276687.txt,14.0,33.0,41.0,25.0,NaN,NaN,NaN,...,3,5,1,48:59.6,FALSE,0,NaN,418,1,20


In [100]:
# Convert the date strings to a uniform datetime format
df['FechaNacimiento'] = pd.to_datetime(df['FechaNacimiento'], format='%d-%m-%Y', errors='coerce').fillna(
    pd.to_datetime(df['FechaNacimiento'], format='%d/%m/%Y', errors='coerce'))

# Reference date (January 1, 2024)
reference_date = datetime(2024, 1, 1)

# Function to calculate age
def calculate_age(birthdate, ref_date):
    return ref_date.year - birthdate.year - ((ref_date.month, ref_date.day) < (birthdate.month, birthdate.day))

# Calculate age and add to new column
df['age_as_of_2024'] = df['FechaNacimiento'].apply(lambda x: calculate_age(x, reference_date))

# Display the result
df['age_as_of_2024']
df['FechaNacimiento']

0       1971-03-25
1       1971-03-25
2       1971-03-25
3       1971-03-25
4       1971-03-25
           ...    
68321   1988-12-30
68322   1988-12-30
68323   1988-12-30
68324   1988-12-30
68325          NaT
Name: FechaNacimiento, Length: 68326, dtype: datetime64[ns]

In [101]:
# export merged file
df.to_csv('merged_truncated.csv')

In [102]:
# data imputation 

col_start = 'Core_universal_dependencies'
col_end = 'Texto_tokenized_sa_NP_skewness_psyco'

# get the list of columns between the start and end columns, inclusive
col_list = df.columns[df.columns.get_loc(col_start) : df.columns.get_loc(col_end) + 1]

# loop through each column in the list
for col in col_list:
    # convert non-numeric entries to NaN
    df[col] = pd.to_numeric(df[col], errors='coerce')
    # compute the median, skipping NaN values
    median_value = df[col].median(skipna=True)
    # replace NaN with the median value
    df[col].fillna(median_value, inplace=True)

In [103]:
# remove missing ages
df = df[df['age_as_of_2024'].notna()]
df['age_as_of_2024']

0        52.0
1        52.0
2        52.0
3        52.0
4        52.0
         ... 
68320    35.0
68321    35.0
68322    35.0
68323    35.0
68324    35.0
Name: age_as_of_2024, Length: 41293, dtype: float64

In [104]:
# Define bins and labels
bins = [0, 18, 23, 30, 40, 60, 70, float('inf')]
labels = ['Under 18', '18-22', '23-29', '30-39', '40-59', '60-69', '70 or above']

# Categorize ages into bins
df['age_category'] = pd.cut(df['age_as_of_2024'], bins=bins, labels=labels, right=False)

# Display the result
print(df['age_category'])

0        40-59
1        40-59
2        40-59
3        40-59
4        40-59
         ...  
68320    30-39
68321    30-39
68322    30-39
68323    30-39
68324    30-39
Name: age_category, Length: 41293, dtype: category
Categories (7, object): ['Under 18' < '18-22' < '23-29' < '30-39' < '40-59' < '60-69' < '70 or above']


/var/folders/qt/wx4bttqn4h13mk9smx_yjchc0000gn/T/ipykernel_81094/991509981.py:6: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['age_category'] = pd.cut(df['age_as_of_2024'], bins=bins, labels=labels, right=False)
/var/folders/qt/wx4bttqn4h13mk9smx_yjchc0000gn/T/ipykernel_81094/991509981.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['age_category'] = pd.cut(df['age_as_of_2024'], bins=bins, labels=labels, right=False)


In [111]:
# remove weird outliers in the education column 

# Convert 'EducacionFormalYYYY' to numeric, converting errors to NaN
df['EducacionFormalYYYY'] = pd.to_numeric(df['EducacionFormalYYYY'], errors='coerce')

# Drop rows with NaN in 'EducacionFormalYYYY'
df = df.dropna(subset=['EducacionFormalYYYY'])

# Convert 'EducacionFormalYYYY' to integer
df['EducacionFormalYYYY'] = df['EducacionFormalYYYY'].astype(int)

# Filter rows where 'EducacionFormalYYYY' <= 50 or are negative
df = df[(df['EducacionFormalYYYY'] >= 0) & (df['EducacionFormalYYYY'] <= 50)]

In [112]:
# CORRELATION BETWEEN AGE AND EDUCATION
corr_s = df['age_as_of_2024'].corr(df['EducacionFormalYYYY'],method='spearman')
print(corr_s)

0.28110339321591493


In [114]:
# FEATURE CORRELATIONS WITH EDUCATION 
# ONLY USING SPEARMAN

# Initialize an empty dictionary to store correlations
correlations = {}

for col in col_list:
    # Ensure both columns are numeric
    df[col] = pd.to_numeric(df[col], errors='coerce')
    df['EducacionFormalYYYY'] = pd.to_numeric(df['EducacionFormalYYYY'], errors='coerce')
    
    # Compute the correlation coefficient, skipping NaN values
    corr = df[col].corr(df['EducacionFormalYYYY'],method='spearman')
    correlations[col] = corr

# Convert the dictionary to a DataFrame for better visualization
correlation_df = pd.DataFrame.from_dict(correlations, orient='index', columns=['Correlation_with_Education'])

# Reset index to turn the index into a column
correlation_df.reset_index(inplace=True)
correlation_df.rename(columns={'index': 'Feature'}, inplace=True)

# Display the correlation DataFrame
print(correlation_df)

correlation_df.to_csv('education_correlations.csv')

                                   Feature  Correlation_with_Education
0              Core_universal_dependencies                    0.010108
1          Non-core_universal_dependencies                   -0.010464
2           Nominal_universal_dependencies                    0.056737
3    Non-dependency_universal_dependencies                    0.032944
4              IdMundialTexto_granularidad                    0.055557
..                                     ...                         ...
106     Texto_tokenized_sa_NP_maximo_psyco                   -0.003973
107        Texto_tokenized_sa_NP_std_psyco                   -0.026883
108    Texto_tokenized_sa_NP_mediana_psyco                   -0.049847
109   Texto_tokenized_sa_NP_curtosis_psyco                    0.025691
110   Texto_tokenized_sa_NP_skewness_psyco                    0.030626

[111 rows x 2 columns]


In [115]:
# EDUCATION

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
import re

# Assuming 'df' is your DataFrame and 'col_list' is already defined

# Ensure that 'EducacionFormalYYYY' is numeric
df['EducacionFormalYYYY'] = pd.to_numeric(df['EducacionFormalYYYY'], errors='coerce')

# Set the style of the plots
sns.set_style('whitegrid')

# Specify the directory where plots will be saved
output_dir = 'scatterplots/'
os.makedirs(output_dir, exist_ok=True)

# Loop through each feature and create a scatterplot
for col in col_list:
    # Ensure the feature column is numeric
    df[col] = pd.to_numeric(df[col], errors='coerce')
    
    # Create a new figure
    plt.figure(figsize=(8, 6))
    
    # Create scatterplot
    sns.scatterplot(
        x='EducacionFormalYYYY',
        y=col,
        data=df,
        alpha=0.7,
        edgecolor=None
    )
    
    # Set title and labels
    plt.title(f'{col} vs Education')
    plt.xlabel('EducacionFormalYYYY')
    plt.ylabel(col)
    
    # Adjust layout
    plt.tight_layout()
    
    # Sanitize the feature name for filename
    safe_col = re.sub(r'[\\/*?:"<>|]', "_", col)
    
    # Define the filename
    filename = os.path.join(output_dir, f'scatterplot_{safe_col}_vs_education.png')
    
    # Save the figure as a PNG file
    plt.savefig(filename, dpi=300)
    print(f'Saved plot to {filename}')
    
    # Close the figure to free up memory
    plt.close()

Saved plot to scatterplots/scatterplot_Core_universal_dependencies_vs_education.png
Saved plot to scatterplots/scatterplot_Non-core_universal_dependencies_vs_education.png
Saved plot to scatterplots/scatterplot_Nominal_universal_dependencies_vs_education.png
Saved plot to scatterplots/scatterplot_Non-dependency_universal_dependencies_vs_education.png
Saved plot to scatterplots/scatterplot_IdMundialTexto_granularidad_vs_education.png
Saved plot to scatterplots/scatterplot_IdMundial_granularidad_vs_education.png
Saved plot to scatterplots/scatterplot_Texto_granularidad_promedio_granularidad_vs_education.png
Saved plot to scatterplots/scatterplot_Texto_granularidad_minimo_granularidad_vs_education.png
Saved plot to scatterplots/scatterplot_Texto_granularidad_maximo_granularidad_vs_education.png
Saved plot to scatterplots/scatterplot_Texto_granularidad_std_granularidad_vs_education.png
Saved plot to scatterplots/scatterplot_Texto_granularidad_skewness_granularidad_vs_education.png
Saved pl

In [116]:
# FEATURE CORRELATIONS WITH AGE
# ONLY USING SPEARMAN
# Initialize an empty dictionary to store correlations
correlations = {}

for col in col_list:
    # Ensure both columns are numeric
    df[col] = pd.to_numeric(df[col], errors='coerce')
    df['age_as_of_2024'] = pd.to_numeric(df['age_as_of_2024'], errors='coerce')
    
    # Compute the correlation coefficient, skipping NaN values
    corr = df[col].corr(df['age_as_of_2024'],method='spearman')
    correlations[col] = corr

# Convert the dictionary to a DataFrame for better visualization
correlation_df = pd.DataFrame.from_dict(correlations, orient='index', columns=['age_as_of_2024'])

# Reset index to turn the index into a column
correlation_df.reset_index(inplace=True)
correlation_df.rename(columns={'index': 'Feature'}, inplace=True)

# Display the correlation DataFrame
print(correlation_df)

correlation_df.to_csv('age_correlations.csv')

                                   Feature  age_as_of_2024
0              Core_universal_dependencies       -0.059274
1          Non-core_universal_dependencies       -0.111641
2           Nominal_universal_dependencies        0.053152
3    Non-dependency_universal_dependencies        0.013032
4              IdMundialTexto_granularidad        0.149802
..                                     ...             ...
106     Texto_tokenized_sa_NP_maximo_psyco       -0.064774
107        Texto_tokenized_sa_NP_std_psyco       -0.086132
108    Texto_tokenized_sa_NP_mediana_psyco       -0.102259
109   Texto_tokenized_sa_NP_curtosis_psyco        0.050507
110   Texto_tokenized_sa_NP_skewness_psyco        0.059310

[111 rows x 2 columns]


In [117]:
# SCATTERPLOTS

In [118]:
# AGE

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
import re

# Assuming 'df' is your DataFrame and 'col_list' is already defined

# Ensure that 'age_as_of_2024' is numeric
df['age_as_of_2024'] = pd.to_numeric(df['age_as_of_2024'], errors='coerce')

# Set the style of the plots
sns.set_style('whitegrid')

# Specify the directory where plots will be saved
output_dir = 'scatterplots/'
os.makedirs(output_dir, exist_ok=True)

# Loop through each feature and create a scatterplot
for col in col_list:
    # Ensure the feature column is numeric
    df[col] = pd.to_numeric(df[col], errors='coerce')
    
    # Create a new figure
    plt.figure(figsize=(8, 6))
    
    # Create scatterplot
    sns.scatterplot(
        x='age_as_of_2024',
        y=col,
        data=df,
        alpha=0.7,
        edgecolor=None
    )
    
    # Set title and labels
    plt.title(f'{col} vs Age')
    plt.xlabel('Age as of 2024')
    plt.ylabel(col)
    
    # Adjust layout
    plt.tight_layout()
    
    # Sanitize the feature name for filename
    safe_col = re.sub(r'[\\/*?:"<>|]', "_", col)
    
    # Define the filename
    filename = os.path.join(output_dir, f'scatterplot_{safe_col}_vs_age.png')
    
    # Save the figure as a PNG file
    plt.savefig(filename, dpi=300)
    print(f'Saved plot to {filename}')
    
    # Close the figure to free up memory
    plt.close()

Saved plot to scatterplots/scatterplot_Core_universal_dependencies_vs_age.png
Saved plot to scatterplots/scatterplot_Non-core_universal_dependencies_vs_age.png
Saved plot to scatterplots/scatterplot_Nominal_universal_dependencies_vs_age.png
Saved plot to scatterplots/scatterplot_Non-dependency_universal_dependencies_vs_age.png
Saved plot to scatterplots/scatterplot_IdMundialTexto_granularidad_vs_age.png
Saved plot to scatterplots/scatterplot_IdMundial_granularidad_vs_age.png
Saved plot to scatterplots/scatterplot_Texto_granularidad_promedio_granularidad_vs_age.png
Saved plot to scatterplots/scatterplot_Texto_granularidad_minimo_granularidad_vs_age.png
Saved plot to scatterplots/scatterplot_Texto_granularidad_maximo_granularidad_vs_age.png
Saved plot to scatterplots/scatterplot_Texto_granularidad_std_granularidad_vs_age.png
Saved plot to scatterplots/scatterplot_Texto_granularidad_skewness_granularidad_vs_age.png
Saved plot to scatterplots/scatterplot_Texto_granularidad_curtosis_granula

In [119]:
df['EducacionFormalYYYY']

0         8
1         8
2         8
3         8
4         8
         ..
68320    20
68321    20
68322    20
68323    20
68324    20
Name: EducacionFormalYYYY, Length: 41105, dtype: int64

In [18]:
# remove missing ages
df = df[df['age_as_of_2024'].notna()]
df['age_as_of_2024']

0        52.0
1        52.0
2        52.0
3        52.0
4        52.0
         ... 
68320    35.0
68321    35.0
68322    35.0
68323    35.0
68324    35.0
Name: age_as_of_2024, Length: 41293, dtype: float64

In [19]:
# Define bins and labels
bins = [0, 18, 23, 30, 40, 60, 70, float('inf')]
labels = ['Under 18', '18-22', '23-29', '30-39', '40-59', '60-69', '70 or above']

# Categorize ages into bins
df['age_category'] = pd.cut(df['age_as_of_2024'], bins=bins, labels=labels, right=False)

# Display the result
print(df['age_category'])

0        40-59
1        40-59
2        40-59
3        40-59
4        40-59
         ...  
68320    30-39
68321    30-39
68322    30-39
68323    30-39
68324    30-39
Name: age_category, Length: 41293, dtype: category
Categories (7, object): ['Under 18' < '18-22' < '23-29' < '30-39' < '40-59' < '60-69' < '70 or above']


/var/folders/qt/wx4bttqn4h13mk9smx_yjchc0000gn/T/ipykernel_81094/991509981.py:6: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['age_category'] = pd.cut(df['age_as_of_2024'], bins=bins, labels=labels, right=False)
/var/folders/qt/wx4bttqn4h13mk9smx_yjchc0000gn/T/ipykernel_81094/991509981.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['age_category'] = pd.cut(df['age_as_of_2024'], bins=bins, labels=labels, right=False)


In [20]:
corr = df['age_as_of_2024'].corr(df['EducacionFormalYYYY'],method='spearman')
corr

np.float64(0.27906108788167483)

In [22]:
# Initialize an empty dictionary to store correlations
correlations = {}

for col in col_list:
    # Ensure both columns are numeric
    df[col] = pd.to_numeric(df[col], errors='coerce')
    df['age_as_of_2024'] = pd.to_numeric(df['age_as_of_2024'], errors='coerce')
    
    # Compute the correlation coefficient, skipping NaN values
    corr = df[col].corr(df['age_as_of_2024'])
    correlations[col] = corr

# Convert the dictionary to a DataFrame for better visualization
correlation_df = pd.DataFrame.from_dict(correlations, orient='index', columns=['Correlation_with_Age'])

# Reset index to turn the index into a column
correlation_df.reset_index(inplace=True)
correlation_df.rename(columns={'index': 'Feature'}, inplace=True)

# Display the correlation DataFrame
print(correlation_df)

correlation_df.to_csv('age_correlations.csv')

                                   Feature  Correlation_with_Age
0              Core_universal_dependencies             -0.053442
1          Non-core_universal_dependencies             -0.071983
2           Nominal_universal_dependencies             -0.006325
3    Non-dependency_universal_dependencies             -0.027945
4              IdMundialTexto_granularidad              0.154372
..                                     ...                   ...
106     Texto_tokenized_sa_NP_maximo_psyco             -0.068891
107        Texto_tokenized_sa_NP_std_psyco             -0.082955
108    Texto_tokenized_sa_NP_mediana_psyco             -0.092207
109   Texto_tokenized_sa_NP_curtosis_psyco              0.064585
110   Texto_tokenized_sa_NP_skewness_psyco              0.069965

[111 rows x 2 columns]


/var/folders/qt/wx4bttqn4h13mk9smx_yjchc0000gn/T/ipykernel_81094/300615567.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df[col] = pd.to_numeric(df[col], errors='coerce')
/var/folders/qt/wx4bttqn4h13mk9smx_yjchc0000gn/T/ipykernel_81094/300615567.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['age_as_of_2024'] = pd.to_numeric(df['age_as_of_2024'], errors='coerce')


In [23]:
# Initialize an empty dictionary to store correlations
correlations = {}

for col in col_list:
    # Ensure both columns are numeric
    df[col] = pd.to_numeric(df[col], errors='coerce')
    df['EducacionFormalYYYY'] = pd.to_numeric(df['EducacionFormalYYYY'], errors='coerce')
    
    # Compute the correlation coefficient, skipping NaN values
    corr = df[col].corr(df['EducacionFormalYYYY'])
    correlations[col] = corr

# Convert the dictionary to a DataFrame for better visualization
correlation_df = pd.DataFrame.from_dict(correlations, orient='index', columns=['Correlation_with_Education'])

# Reset index to turn the index into a column
correlation_df.reset_index(inplace=True)
correlation_df.rename(columns={'index': 'Feature'}, inplace=True)

# Display the correlation DataFrame
print(correlation_df)

correlation_df.to_csv('education_correlations.csv')

                                   Feature  Correlation_with_Education
0              Core_universal_dependencies                    0.001910
1          Non-core_universal_dependencies                   -0.000221
2           Nominal_universal_dependencies                    0.002839
3    Non-dependency_universal_dependencies                    0.001106
4              IdMundialTexto_granularidad                    0.003719
..                                     ...                         ...
106     Texto_tokenized_sa_NP_maximo_psyco                    0.004655
107        Texto_tokenized_sa_NP_std_psyco                   -0.005239
108    Texto_tokenized_sa_NP_mediana_psyco                   -0.000787
109   Texto_tokenized_sa_NP_curtosis_psyco                    0.009380
110   Texto_tokenized_sa_NP_skewness_psyco                    0.008162

[111 rows x 2 columns]


/var/folders/qt/wx4bttqn4h13mk9smx_yjchc0000gn/T/ipykernel_81094/1663982872.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df[col] = pd.to_numeric(df[col], errors='coerce')
/var/folders/qt/wx4bttqn4h13mk9smx_yjchc0000gn/T/ipykernel_81094/1663982872.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['EducacionFormalYYYY'] = pd.to_numeric(df['EducacionFormalYYYY'], errors='coerce')


In [9]:
# Function to replace non-numeric values with NaN and then fill NaNs with column mean
def clean_column(col):
    col = pd.to_numeric(col, errors='coerce')
    return col.fillna(col.mean())

In [ ]:
# Optional: If your dataset has a target column, separate it
target = df_WWC['filename']
data = data.drop('target_column_name', axis=1)

In [11]:
# Apply the function to each column in the dataframe
df_WWC = df_WWC.apply(clean_column)

In [14]:
# Standardizing the Data (important for PCA)
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
scaled_data = scaler.fit_transform(df_WWC)

/Users/jpd/Documents/env/lib/python3.9/site-packages/sklearn/utils/extmath.py:1140: RuntimeWarning: invalid value encountered in divide
  updated_mean = (last_sum + new_sum) / updated_sample_count
/Users/jpd/Documents/env/lib/python3.9/site-packages/sklearn/utils/extmath.py:1145: RuntimeWarning: invalid value encountered in divide
  T = new_sum / new_sample_count
/Users/jpd/Documents/env/lib/python3.9/site-packages/sklearn/utils/extmath.py:1165: RuntimeWarning: invalid value encountered in divide
  new_unnormalized_variance -= correction**2 / new_sample_count


In [15]:
# Performing PCA
pca = PCA(n_components=2)  # Change n_components to the number of principal components you want
principal_components = pca.fit_transform(scaled_data)

ValueError: Input X contains NaN.
PCA does not accept missing values encoded as NaN natively. For supervised learning, you might want to consider sklearn.ensemble.HistGradientBoostingClassifier and Regressor which accept missing values encoded as NaNs natively. Alternatively, it is possible to preprocess the data, for instance by using an imputer transformer in a pipeline or drop samples with missing values. See https://scikit-learn.org/stable/modules/impute.html You can find a list of all estimators that handle NaN values at the following page: https://scikit-learn.org/stable/modules/impute.html#estimators-that-handle-nan-values

In [ ]:
import pandas as pd
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
import numpy as np

# Load the dataset
# Replace 'your_dataset.csv' with your actual file path
data = pd.read_csv('your_dataset.csv')

# Function to replace non-numeric values with NaN and then fill NaNs with column mean
def clean_column(col):
    col = pd.to_numeric(col, errors='coerce')
    return col.fillna(col.mean())

# Apply the function to each column in the dataframe
data = data.apply(clean_column)

# Optional: If your dataset has a target column, separate it
# target = data['target_column_name']
# data = data.drop('target_column_name', axis=1)

# Standardizing the Data (important for PCA)
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
scaled_data = scaler.fit_transform(data)

# Performing PCA
pca = PCA(n_components=2)  # Change n_components to the number of principal components you want
principal_components = pca.fit_transform(scaled_data)

# Creating a DataFrame with the principal components
pc_df = pd.DataFrame(data=principal_components, columns=['PC1', 'PC2'])

# Optional: Adding the target column back if it was separated earlier
# pc_df = pd.concat([pc_df, target], axis=1)

# Plotting the principal components
plt.figure(figsize=(8, 6))
plt.scatter(pc_df['PC1'], pc_df['PC2'])
plt.xlabel('Principal Component 1')
plt.ylabel('Principal Component 2')
plt.title('2D PCA')
plt.grid()
plt.show()

# Explained variance
print('Explained variance ratio:', pca.explained_variance_ratio_)
print('Singular values:', pca.singular_values_)

# Save the principal components to a CSV file
pc_df.to_csv('principal_components.csv', index=False)


In [9]:
df_WWC.FRASE_readability.dropna()

0        2.0
1        4.0
2        2.0
3        3.0
4        3.0
        ... 
98158    2.0
98159    2.0
98160    2.0
98161    2.0
98162    2.0
Name: FRASE_readability, Length: 98163, dtype: float64

In [10]:
df_WWC.FRASE_readability

0        2.0
1        4.0
2        2.0
3        3.0
4        3.0
        ... 
99233    NaN
99234    NaN
99235    NaN
99236    NaN
99237    NaN
Name: FRASE_readability, Length: 99238, dtype: float64

In [26]:
## Constants

CORPUS_PATH = 'dataset_as_files'
STOPWORDS = stopwords.words('spanish')

In [27]:
# Change to glob

files = [file for file in os.listdir(CORPUS_PATH) if file.endswith('.txt')]

In [28]:
def preprocess_text(text):
    # Helper function for preprocessing text
    text = text.lower()
    text = re.sub('[^a-zA-Z]', ' ', text)
    text = re.sub(r'\s+', ' ', text)
    return text

In [29]:
def stream_texts(files, preprocess=True):
    # Helper function to stream and preprocessed corpus
    for file in files:
        with open(f'{CORPUS_PATH}/{file}', 'r') as f:
            contents = f.read()
            sents = nltk.sent_tokenize(contents)
            if preprocess:
                sents = [preprocess_text(sent) for sent in sents]
            token_sents = [nltk.word_tokenize(sent) for sent in sents]
            token_sents = [[token for token in token_sent if token not in STOPWORDS] for token_sent in token_sents]
            yield token_sents

In [30]:
class SentenceIterator:
    # Cf. https://stackoverflow.com/a/55091252
    def __init__(self, path, preprocess=True):
        self.path = path
        self.preprocess = preprocess

    def __iter__(self):
        files = [file for file in os.listdir(self.path) if file.endswith('.txt')]
        for file in files:
            with open(f'{self.path}/{file}', 'r') as f:
                contents = f.read()
                sents = nltk.sent_tokenize(contents)
                if self.preprocess:
                    sents = [preprocess_text(sent) for sent in sents]
                token_sents = [nltk.word_tokenize(sent) for sent in sents]
                token_sents = [[token for token in token_sent if token not in STOPWORDS] for token_sent in token_sents]
                for token_sent in token_sents:
                    yield token_sent

In [31]:
sentences = SentenceIterator(CORPUS_PATH)
w2v_model = Word2Vec(sentences, vector_size=100, window=5, min_count=5)
w2v_model.save('wwc_w2v_size100_window5_min5.model')